In [ ]:
import random
from dataclasses import dataclass, field

import numpy as np
import torch.nn.functional as F
from torch import optim
from torch.distributions import Categorical
from torch.utils.data import DataLoader, TensorDataset

import traci
import config
from Action import Action
from ppo.PPO_actor_critic import Actor, Critic
from config import *
from logger_utils import EpisodeLogger, default_episode_log_path
from sumo_utils import (
    apply_action,
    compute_reward,
    ego_exists,
    get_state,
    is_abnormal_disappearance,
    is_arrived,
    reset_sumo,
    spawn_ego,
    start_sumo,
)

OBS_SIZE = 12
N_ACTIONS = len(Action)

PPO_EPOCHS = 4
PPO_BATCH_SIZE = 64
PPO_CLIP_EPS = 0.2
GAE_LAMBDA = 0.95
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5
MAX_GRAD_NORM = 0.5
CHECKPOINT_FREQ = 50

# Reset counters so notebook reruns start cleanly.
config.TOTAL_EGO_CRASHES = 0
config.TOTAL_COLLISION_EVENTS = 0
config.TOTAL_EGO_COLLISIONS = 0
config.TOTAL_EGO_TELEPORTS = 0
config.TOTAL_EGO_EMERGENCY_STOPS = 0


In [ ]:
@dataclass
class RolloutBuffer:
    states: list = field(default_factory=list)
    actions: list = field(default_factory=list)
    log_probs: list = field(default_factory=list)
    rewards: list = field(default_factory=list)
    dones: list = field(default_factory=list)
    values: list = field(default_factory=list)

    def add(self, state, action, log_prob, reward, done, value):
        self.states.append(np.asarray(state, dtype=np.float32))
        self.actions.append(int(action))
        self.log_probs.append(float(log_prob))
        self.rewards.append(float(reward))
        self.dones.append(float(done))
        self.values.append(float(value))

    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.log_probs.clear()
        self.rewards.clear()
        self.dones.clear()
        self.values.clear()

    def __len__(self):
        return len(self.states)

@torch.no_grad()
def select_action(actor, critic, state, deterministic=False):
    state_t = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    logits = actor(state_t)
    dist = Categorical(logits=logits)
    action_t = torch.argmax(logits, dim=-1) if deterministic else dist.sample()
    log_prob = dist.log_prob(action_t)
    value = critic(state_t).squeeze(-1)
    return (
        int(action_t.item()),
        float(log_prob.item()),
        float(value.item()),
        float(dist.entropy().item()),
    )


def compute_gae(rewards, dones, values, last_value, gamma=GAMMA, gae_lambda=GAE_LAMBDA):
    rewards = np.asarray(rewards, dtype=np.float32)
    dones = np.asarray(dones, dtype=np.float32)
    values = np.asarray(values + [last_value], dtype=np.float32)

    advantages = np.zeros_like(rewards, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(len(rewards))):
        mask = 1.0 - dones[t]
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]
        gae = delta + gamma * gae_lambda * mask * gae
        advantages[t] = gae

    returns = advantages + values[:-1]
    return returns, advantages


def update_safety_counters():
    colliding_ids = traci.simulation.getCollidingVehiclesIDList()
    collision_events = traci.simulation.getCollisions()
    teleport_ids = traci.simulation.getStartingTeleportIDList()
    emergency_ids = traci.simulation.getEmergencyStoppingVehiclesIDList()

    config.TOTAL_COLLISION_EVENTS += len(collision_events)

    ego_collision = EGO_ID in colliding_ids
    ego_teleport = EGO_ID in teleport_ids
    ego_emergency = EGO_ID in emergency_ids

    if ego_collision:
        config.TOTAL_EGO_COLLISIONS += 1
    if ego_teleport:
        config.TOTAL_EGO_TELEPORTS += 1
    if ego_emergency:
        config.TOTAL_EGO_EMERGENCY_STOPS += 1
    if ego_collision or ego_teleport or ego_emergency:
        config.TOTAL_EGO_CRASHES += 1

    return ego_collision


def ppo_update(actor, critic, optimizer, rollout, last_value=0.0):
    if len(rollout) == 0:
        return {
            'policy_loss': 0.0,
            'value_loss': 0.0,
            'entropy': 0.0,
            'loss': 0.0,
        }

    states = torch.as_tensor(np.asarray(rollout.states), dtype=torch.float32, device=DEVICE)
    actions = torch.as_tensor(rollout.actions, dtype=torch.long, device=DEVICE)
    old_log_probs = torch.as_tensor(rollout.log_probs, dtype=torch.float32, device=DEVICE)
    returns_np, advantages_np = compute_gae(
        rollout.rewards,
        rollout.dones,
        rollout.values,
        last_value,
    )
    returns = torch.as_tensor(returns_np, dtype=torch.float32, device=DEVICE)
    advantages = torch.as_tensor(advantages_np, dtype=torch.float32, device=DEVICE)

    if advantages.numel() > 1:
        advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)

    dataset = TensorDataset(states, actions, old_log_probs, returns, advantages)
    batch_size = min(PPO_BATCH_SIZE, len(dataset))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    policy_losses = []
    value_losses = []
    entropies = []
    total_losses = []

    params = list(actor.parameters()) + list(critic.parameters())

    for _ in range(PPO_EPOCHS):
        for batch_states, batch_actions, batch_old_log_probs, batch_returns, batch_advantages in loader:
            logits = actor(batch_states)
            dist = Categorical(logits=logits)
            new_log_probs = dist.log_prob(batch_actions)
            entropy = dist.entropy().mean()
            values_pred = critic(batch_states).squeeze(-1)

            ratio = torch.exp(new_log_probs - batch_old_log_probs)
            surr1 = ratio * batch_advantages
            surr2 = torch.clamp(ratio, 1.0 - PPO_CLIP_EPS, 1.0 + PPO_CLIP_EPS) * batch_advantages
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss = F.mse_loss(values_pred, batch_returns)
            loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
            optimizer.step()

            policy_losses.append(float(policy_loss.item()))
            value_losses.append(float(value_loss.item()))
            entropies.append(float(entropy.item()))
            total_losses.append(float(loss.item()))

    return {
        'policy_loss': float(np.mean(policy_losses)) if policy_losses else 0.0,
        'value_loss': float(np.mean(value_losses)) if value_losses else 0.0,
        'entropy': float(np.mean(entropies)) if entropies else 0.0,
        'loss': float(np.mean(total_losses)) if total_losses else 0.0,
    }


In [ ]:
def run_ppo_episode(actor, critic, optimizer, global_step):
    rollout = RolloutBuffer()
    route_id = random.choice(EGO_ROUTE_POOL)

    reset_sumo()
    spawn_ego(route_id)

    # Let SUMO advance one step so the vehicle is fully inserted.
    traci.simulationStep()

    if not ego_exists():
        return 0.0, 0, global_step, 'spawn_failed', {
            'policy_loss': 0.0,
            'value_loss': 0.0,
            'entropy': 0.0,
            'loss': 0.0,
        }

    episode_reward = 0.0
    end_reason = 'timeout'
    state = get_state(EGO_ID)
    steps_taken = 0
    last_value = 0.0

    for step in range(MAX_STEPS_PER_EPISODE):
        action_idx, log_prob, value, _ = select_action(actor, critic, state)
        delta_v = apply_action(EGO_ID, Action(action_idx))

        traci.simulationStep()
        global_step += 1
        steps_taken = step + 1

        ego_collision = update_safety_counters()

        if ego_collision:
            reward = -30.0
            rollout.add(state, action_idx, log_prob, reward, 1.0, value)
            episode_reward += reward
            end_reason = 'ego_crash'
            last_value = 0.0
            break

        if is_arrived():
            reward = 20.0
            rollout.add(state, action_idx, log_prob, reward, 1.0, value)
            episode_reward += reward
            end_reason = 'arrived'
            last_value = 0.0
            break

        if is_abnormal_disappearance():
            reward = -20.0
            rollout.add(state, action_idx, log_prob, reward, 1.0, value)
            episode_reward += reward
            end_reason = 'abnormal_end'
            last_value = 0.0
            break

        next_state = get_state(EGO_ID)
        reward = compute_reward(EGO_ID, delta_v)
        rollout.add(state, action_idx, log_prob, reward, 0.0, value)

        episode_reward += reward
        state = next_state
    else:
        if ego_exists():
            last_value = float(
                critic(
                    torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                ).item()
            )
        else:
            last_value = 0.0

    stats = ppo_update(actor, critic, optimizer, rollout, last_value=last_value)
    return episode_reward, steps_taken, global_step, end_reason, stats


In [ ]:
start_sumo()

actor = Actor(OBS_SIZE, N_ACTIONS).to(DEVICE)
critic = Critic(OBS_SIZE).to(DEVICE)
optimizer = optim.Adam(
    list(actor.parameters()) + list(critic.parameters()),
    lr=LR,
)

global_step = 0
episode_logger = EpisodeLogger(default_episode_log_path())
print(f'Logging episodes to: {episode_logger.path}')

try:
    for episode in range(NUM_EPISODES):
        ep_reward, ep_steps, global_step, end_reason, stats = run_ppo_episode(
            actor,
            critic,
            optimizer,
            global_step,
        )

        print(
            f'Episode {episode:4d} | '
            f'reward={ep_reward:8.2f} | '
            f'steps={ep_steps:4d} | '
            f'end={end_reason} | '
            f'policy_loss={stats["policy_loss"]:.4f} | '
            f'value_loss={stats["value_loss"]:.4f} | '
            f'entropy={stats["entropy"]:.4f} | '
            f'TOTAL_EGO_CRASHES={config.TOTAL_EGO_CRASHES} | '
            f'TOTAL_COLLISION_EVENTS={config.TOTAL_COLLISION_EVENTS} | '
            f'TOTAL_EGO_COLLISIONS={config.TOTAL_EGO_COLLISIONS} | '
            f'TOTAL_EGO_TELEPORTS={config.TOTAL_EGO_TELEPORTS} | '
            f'TOTAL_EGO_EMERGENCY_STOPS={config.TOTAL_EGO_EMERGENCY_STOPS}'
        )

        episode_logger.log(
            episode=episode,
            reward=ep_reward,
            steps=ep_steps,
            end_reason=end_reason,
            total_ego_crashes=config.TOTAL_EGO_CRASHES,
            total_collision_events=config.TOTAL_COLLISION_EVENTS,
            total_ego_collisions=config.TOTAL_EGO_COLLISIONS,
            total_ego_teleports=config.TOTAL_EGO_TELEPORTS,
            total_ego_emergency_stops=config.TOTAL_EGO_EMERGENCY_STOPS,
        )

        if (episode + 1) % CHECKPOINT_FREQ == 0:
            checkpoint_path = f'ppo_ego_episode_{episode + 1}.pth'
            torch.save(
                {
                    'episode': episode + 1,
                    'global_step': global_step,
                    'actor_state_dict': actor.state_dict(),
                    'critic_state_dict': critic.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                },
                checkpoint_path,
            )
            print(f'Saved checkpoint to {checkpoint_path}')
finally:
    try:
        traci.close()
    except Exception:
        pass


In [3]:
import torch
CHECKPOINT_PATH = "../ppo_training/ppo_ego_episode_500.pth"
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

print(checkpoint.keys())

dict_keys(['episode', 'global_step', 'update_idx', 'actor_state_dict', 'critic_state_dict', 'optimizer_state_dict'])
